# 🚀 TDS Portal Solver - Extended Edition

## Credits & Acknowledgements

This Google Colab notebook is an extended implementation based on the ideas, workflows, and resources developed for **TDS Portal Solver**.

### Creator

**GT_Indian (Gaurav Tomar)**

- 🌐 Official Project: https://tds-portal-solver.vercel.app/
- 💼 LinkedIn: https://www.linkedin.com/in/gaurav-tomar-630b2a316
- 💻 GitHub: https://github.com/GyaanFlow

### About the Original Project

**TDS Portal Solver** is an educational project designed to assist learners in understanding and solving course-related tasks efficiently through automation, data analysis, and AI-powered workflows.

This notebook extends the original concepts by providing additional features, optimizations, experiments, and/or research implementations while preserving credit to the original creator.

### Attribution

If you use, modify, share, or build upon this notebook, please provide appropriate attribution to:

> **GT_Indian (Gaurav Tomar)**  
> TDS Portal Solver  
> https://tds-portal-solver.vercel.app/

### Disclaimer

This notebook is intended for educational and research purposes only. Users are responsible for ensuring compliance with the policies, academic guidelines, and terms of service of any platform or course associated with its usage.

---

⭐ If you find this work useful, consider supporting the project by visiting the website and following the creator on GitHub and LinkedIn.

# TDS Exam — Q10 & Q18 & Q25 Combined Dynamic Solution

## ✅ Works for every student — just edit Cell 1

| Question | Title | What's unique per student |
|----------|-------|---------------------------|
| **Q10** | Write a FastAPI server to serve student data | Your `q-fastapi.csv` (seeded from your email) |
| **Q18** | Local Ollama Endpoint via ngrok | Your email in `X-Email` header + your ngrok token |

### Steps:
1. **Edit Cell 1** — fill in your email and ngrok token
2. **Run Cell 2** — installs all dependencies (one time)
3. **For Q10** — Run Cells 3 → 4 → 5, then copy the `/api` URL to the exam
4. **For Q18** — Run Cells 6 → 7 → 8 → 9, then copy the ngrok base URL to the exam

> Get your ngrok token free at https://dashboard.ngrok.com

In [ ]:
# ╔══════════════════════════════════════════════════════════╗
# ║  CELL 1 — ✏️  EDIT THESE VALUES — then run all cells   ║
# ╚══════════════════════════════════════════════════════════╝

YOUR_EMAIL  = "23fxxxxxx@ds.study.iitm.ac.in"   # <-- your IITM exam email
NGROK_TOKEN = "your-ngrok-token"          # <-- from https://dashboard.ngrok.com

# --- Quick validation ---
assert '@' in YOUR_EMAIL and 'iitm.ac.in' in YOUR_EMAIL, \
    '❌ Set YOUR_EMAIL to your real IITM exam email'
assert len(NGROK_TOKEN) > 20 and NGROK_TOKEN != 'your_ngrok_authtoken_here', \
    '❌ Set NGROK_TOKEN to your real ngrok authtoken'

print(f'✅ Email  : {YOUR_EMAIL}')
print(f'✅ Token  : {NGROK_TOKEN[:8]}...{NGROK_TOKEN[-4:]} (masked)')
print('\nReady! Now run the cells for whichever question you need.')

In [ ]:
# ╔══════════════════════════════════════════════════════════╗
# ║  CELL 2 — Install ALL dependencies (run once)          ║
# ╚══════════════════════════════════════════════════════════╝

import subprocess
subprocess.run(['apt-get', 'update', '-qq'], check=True)
subprocess.run(['apt-get', 'install', '-y', '-qq', 'curl', 'tar', 'zstd'], check=True)
subprocess.run(['pip', 'install', '-q', 'fastapi', 'uvicorn[standard]', 'requests'], check=True)
print('✅ All dependencies installed')

---
# 📦 Q10 — FastAPI Student Data Server

**What the exam checks:** Calls `GET /api?class=1A&class=2B...` and verifies the returned students match your personal CSV — in the same row order as the file.

**Why your CSV is unique:** It is generated from your email as a random seed. Download it from the exam portal (there is a download button in the Q10 question).

Run **Cells 3 → 4 → 5** in order.

In [ ]:
# ╔══════════════════════════════════════════════════════════╗
# ║  CELL 3 (Q10) — Upload your q-fastapi.csv              ║
# ╚══════════════════════════════════════════════════════════╝
# Download q-fastapi.csv from the exam portal first,
# then upload it here.

from google.colab import files
from pathlib import Path
import csv

CSV_NAME = 'q-fastapi.csv'

if Path(CSV_NAME).exists():
    print(f'⚠️  {CSV_NAME} already exists. Delete and re-run to replace.')
else:
    print(f'📂 Please upload your personal {CSV_NAME} from the exam portal...')
    uploaded = files.upload()
    if CSV_NAME not in uploaded:
        for fname in uploaded:
            if fname.endswith('.csv'):
                Path(fname).rename(CSV_NAME)
                print(f'   Renamed {fname} → {CSV_NAME}')
                break
        else:
            raise SystemExit('❌ Upload a file named q-fastapi.csv')

with open(CSV_NAME) as f:
    rows = list(csv.DictReader(f))

print(f'\n✅ Loaded {len(rows)} students')
print('First 3 rows :', rows[:3])
print('Sample classes:', list({r["class"] for r in rows})[:8])

In [ ]:
# ╔══════════════════════════════════════════════════════════╗
# ║  CELL 4 (Q10) — Write and start FastAPI server         ║
# ╚══════════════════════════════════════════════════════════╝

import subprocess, os, time, requests
from pathlib import Path

os.system('fuser -k 8001/tcp 2>/dev/null || true')
time.sleep(0.5)

server_code = '''
from fastapi import FastAPI, Query
from fastapi.middleware.cors import CORSMiddleware
from typing import List, Optional
import csv
from pathlib import Path

app = FastAPI(title="q-fastapi Student API")

app.add_middleware(
    CORSMiddleware,
    allow_origins=["*"],
    allow_methods=["GET", "OPTIONS"],
    allow_headers=["*"],
)

def load_students():
    students = []
    with Path("q-fastapi.csv").open(newline="", encoding="utf-8") as f:
        for row in csv.DictReader(f):
            try:
                sid = int(row["studentId"])
            except Exception:
                sid = row["studentId"]
            students.append({"studentId": sid, "class": row["class"]})
    return students

STUDENTS = load_students()

@app.get("/api")
async def api(class_: Optional[List[str]] = Query(None, alias="class")):
    if class_ is None:
        return {"students": STUDENTS}
    allowed = set(class_)
    return {"students": [s for s in STUDENTS if s["class"] in allowed]}

@app.get("/")
async def root():
    return {"message": "Student API — use GET /api"}
'''

Path('server.py').write_text(server_code, encoding='utf-8')
print('✅ server.py written')

log_f = open('uvicorn.log', 'w')
uv_proc = subprocess.Popen(
    ['uvicorn', 'server:app', '--host', '0.0.0.0', '--port', '8001'],
    stdout=log_f, stderr=subprocess.STDOUT, preexec_fn=os.setsid
)

for _ in range(20):
    try:
        if requests.get('http://127.0.0.1:8001/').status_code == 200:
            break
    except Exception:
        time.sleep(0.5)

data = requests.get('http://127.0.0.1:8001/api').json()
print(f'✅ Server running — /api returns {len(data["students"])} students')
print(f'   First student: {data["students"][0]}')
print(f'\nUvicorn PID: {uv_proc.pid}')

In [ ]:
# ╔══════════════════════════════════════════════════════════╗
# ║  CELL 5 (Q10) — Create public tunnel (cloudflared)     ║
# ╚══════════════════════════════════════════════════════════╝

import subprocess, os, time, re, requests
from pathlib import Path

if not Path('cloudflared').exists():
    print('Downloading cloudflared...')
    os.system('curl -fsSL -o cloudflared https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64')
    os.chmod('cloudflared', 0o755)
    print('✅ cloudflared downloaded')

os.system("pkill -f 'cloudflared tunnel' || true")
time.sleep(1)

cf_log = open('cloudflared.log', 'w')
cf_proc = subprocess.Popen(
    ['./cloudflared', 'tunnel', '--url', 'http://127.0.0.1:8001'],
    stdout=cf_log, stderr=subprocess.STDOUT, preexec_fn=os.setsid
)

public_url = None
start = time.time()
print('Waiting for tunnel', end='')
while time.time() - start < 45:
    time.sleep(1.5)
    try:
        content = open('cloudflared.log').read()
        m = re.search(r'https://[A-Za-z0-9\-]+\.trycloudflare\.com', content)
        if m:
            public_url = m.group(0)
            break
    except Exception:
        pass
    print('.', end='', flush=True)
print()

if not public_url:
    print('❌ No tunnel URL. cloudflared.log:')
    print(open('cloudflared.log').read()[-1500:])
else:
    time.sleep(2)
    try:
        r = requests.get(f'{public_url}/api', timeout=15)
        count = len(r.json().get('students', []))
        print(f'✅ Public /api returns {count} students')
    except Exception as e:
        print(f'⚠️  Public test: {e} (tunnel may need a moment)')
    print(f'\n🎉 Submit this URL to the exam (Q10):')
    print(f'\n   👉  {public_url}/api')
    print(f'\n   Example: {public_url}/api?class=1A&class=2B')
    print(f'\nCloudflared PID: {cf_proc.pid}')

---
# 🤖 Q18 — Local Ollama Endpoint via ngrok

**What the exam checks:**
- URL must be an ngrok domain
- `GET /api/version` returns JSON with a `version` key (Ollama is running)
- Response header `X-Email` must equal **your exact exam email**

**Why it's dynamic:** The proxy injects `X-Email` from `YOUR_EMAIL` set in Cell 1.

Run **Cells 6 → 7 → 8 → 9** in order.

In [ ]:
# ╔══════════════════════════════════════════════════════════╗
# ║  CELL 6 (Q18) — Install and start Ollama               ║
# ╚══════════════════════════════════════════════════════════╝

import subprocess, os, time, requests

def run(cmd):
    print(f'$ {cmd}')
    subprocess.run(cmd, shell=True, check=False)

run('pkill -f ollama || true')
time.sleep(2)
run('rm -f /usr/local/bin/ollama /usr/bin/ollama || true')
run('rm -rf /usr/lib/ollama /tmp/ollama* || true')

print('\nDownloading Ollama (this takes ~1 min)...')
run('curl -fsSL https://ollama.com/download/ollama-linux-amd64.tar.zst -o /tmp/ollama-linux-amd64.tar.zst')
run('mkdir -p /tmp/ollama-extract')
run('tar --use-compress-program=unzstd -xf /tmp/ollama-linux-amd64.tar.zst -C /tmp/ollama-extract')
run('cp -r /tmp/ollama-extract/* /usr/')
run('chmod +x /usr/bin/ollama || true')
run('/usr/bin/ollama -v')

OLLAMA_LOG = '/content/ollama.log'
env = os.environ.copy()
env['OLLAMA_HOST'] = '127.0.0.1:11434'
env['OLLAMA_ORIGINS'] = '*'

with open(OLLAMA_LOG, 'w') as f:
    ollama_proc = subprocess.Popen(
        ['/usr/bin/ollama', 'serve'],
        stdout=f, stderr=f, env=env, preexec_fn=os.setsid
    )

print('\nWaiting for Ollama...', end='')
for _ in range(60):
    try:
        r = requests.get('http://127.0.0.1:11434/api/version', timeout=3)
        if 'version' in r.json():
            print(f'\n✅ Ollama running: {r.json()}')
            break
    except Exception:
        pass
    print('.', end='', flush=True)
    time.sleep(2)
else:
    print('\n❌ Ollama failed. Log:')
    print(open(OLLAMA_LOG).read()[:2000])

print(f'\nOllama PID: {ollama_proc.pid}')

In [ ]:
# ╔══════════════════════════════════════════════════════════╗
# ║  CELL 7 (Q18) — Start proxy that injects X-Email       ║
# ╚══════════════════════════════════════════════════════════╝
# YOUR_EMAIL from Cell 1 is written into the proxy file.
# This is what makes each student's solution unique.

import subprocess, os, time, requests
from pathlib import Path

# Build proxy source using string concatenation — no f-string curlies
proxy_lines = [
    'import requests',
    'from fastapi import FastAPI, Request, Response',
    'from fastapi.middleware.cors import CORSMiddleware',
    '',
    'OLLAMA_BASE = "http://127.0.0.1:11434"',
    f'YOUR_EMAIL  = {repr(YOUR_EMAIL)}',
    '',
    'app = FastAPI()',
    'app.add_middleware(',
    '    CORSMiddleware, allow_origins=["*"], allow_credentials=False,',
    '    allow_methods=["*"], allow_headers=["*"], expose_headers=["*"],',
    ')',
    '',
    'def extra_headers():',
    '    return {',
    '        "X-Email": YOUR_EMAIL,',
    '        "Access-Control-Allow-Origin": "*",',
    '        "Access-Control-Expose-Headers": "*",',
    '        "Access-Control-Allow-Headers": "Authorization,Content-Type,User-Agent,Accept,Ngrok-Skip-Browser-Warning",',
    '        "Access-Control-Allow-Methods": "GET,POST,PUT,PATCH,DELETE,OPTIONS,HEAD"',
    '    }',
    '',
    '@app.options("/{path:path}")',
    'async def preflight(path: str, request: Request):',
    '    return Response(status_code=200, headers=extra_headers())',
    '',
    '@app.api_route("/{path:path}", methods=["GET","POST","PUT","PATCH","DELETE","OPTIONS","HEAD"])',
    'async def proxy(path: str, request: Request):',
    '    if request.method == "OPTIONS":',
    '        return Response(status_code=200, headers=extra_headers())',
    '    body = await request.body()',
    '    headers = {k: v for k, v in request.headers.items()',
    '               if k.lower() not in {"host", "content-length"}}',
    '    upstream = requests.request(',
    '        method=request.method,',
    '        url=f"{OLLAMA_BASE}/{path}",',
    '        params=list(request.query_params.multi_items()),',
    '        headers=headers, data=body, timeout=300',
    '    )',
    '    excluded = {"content-encoding", "transfer-encoding", "connection"}',
    '    out_headers = {k: v for k, v in upstream.headers.items() if k.lower() not in excluded}',
    '    out_headers.update(extra_headers())',
    '    return Response(',
    '        content=upstream.content, status_code=upstream.status_code,',
    '        headers=out_headers, media_type=upstream.headers.get("content-type")',
    '    )',
]

Path('/content/proxy_app.py').write_text('\n'.join(proxy_lines), encoding='utf-8')
print('✅ proxy_app.py written with email:', YOUR_EMAIL)

os.system('fuser -k 8000/tcp 2>/dev/null || true')
time.sleep(0.5)

PROXY_LOG = '/content/proxy.log'
with open(PROXY_LOG, 'w') as f:
    proxy_proc = subprocess.Popen(
        ['python', '-m', 'uvicorn', 'proxy_app:app', '--host', '0.0.0.0', '--port', '8000'],
        cwd='/content', stdout=f, stderr=f, preexec_fn=os.setsid
    )

print('Waiting for proxy...', end='')
for _ in range(30):
    try:
        r = requests.get('http://127.0.0.1:8000/api/version', timeout=3)
        if 'version' in r.json():
            print(f'\n✅ Proxy working')
            print(f'   X-Email : {r.headers.get("x-email")}')
            print(f'   version : {r.json()}')
            break
    except Exception:
        pass
    print('.', end='', flush=True)
    time.sleep(1)
else:
    print('\n❌ Proxy failed. Log:')
    print(open(PROXY_LOG).read()[:2000])

print(f'\nProxy PID: {proxy_proc.pid}')

In [ ]:
# ╔══════════════════════════════════════════════════════════╗
# ║  CELL 8 (Q18) — Install ngrok and create tunnel        ║
# ╚══════════════════════════════════════════════════════════╝

import shutil, subprocess, os, time, requests

if not shutil.which('ngrok'):
    print('Installing ngrok...')
    subprocess.run(
        'curl -sSL https://bin.equinox.io/c/bNyj1mQVY4c/ngrok-v3-stable-linux-amd64.tgz -o /tmp/ngrok.tgz'
        ' && tar -xzf /tmp/ngrok.tgz -C /tmp'
        ' && mv /tmp/ngrok /usr/local/bin/ngrok'
        ' && chmod +x /usr/local/bin/ngrok',
        shell=True, check=True
    )
    print('✅ ngrok installed')

subprocess.run(['ngrok', 'config', 'add-authtoken', NGROK_TOKEN], check=True)

os.system("pkill -f 'ngrok http' || true")
time.sleep(2)

NGROK_LOG = '/content/ngrok.log'
with open(NGROK_LOG, 'w') as f:
    ngrok_proc = subprocess.Popen(
        ['ngrok', 'http', '8000'],
        stdout=f, stderr=f, preexec_fn=os.setsid
    )

public_url = None
start = time.time()
print('Waiting for ngrok tunnel...', end='')
while time.time() - start < 90:
    try:
        t = requests.get('http://127.0.0.1:4040/api/tunnels', timeout=5)
        https = [x for x in t.json().get('tunnels', []) if x.get('public_url', '').startswith('https://')]
        if https:
            public_url = https[0]['public_url']
            break
    except Exception:
        pass
    print('.', end='', flush=True)
    time.sleep(2)
print()

if not public_url:
    print('❌ No ngrok URL. Log:')
    print(open(NGROK_LOG).read()[:2000])
else:
    print(f'✅ ngrok tunnel ready: {public_url}')
    print(f'   ngrok PID: {ngrok_proc.pid}')

In [ ]:
# ╔══════════════════════════════════════════════════════════╗
# ║  CELL 9 (Q18) — Verify and print final URL             ║
# ╚══════════════════════════════════════════════════════════╝

import requests, time
time.sleep(2)

resp = requests.get(
    f'{public_url}/api/version',
    headers={'ngrok-skip-browser-warning': 'true'},
    timeout=20
)

email_got  = resp.headers.get('x-email', '')
version_ok = 'version' in resp.json()
email_ok   = email_got == YOUR_EMAIL
cors_ok    = resp.headers.get('access-control-allow-origin') == '*'

print('=== FINAL VERIFICATION ===')
print(f'HTTP status          : {resp.status_code}')
print(f'version in body      : {"✅" if version_ok else "❌"}')
print(f'X-Email header       : {email_got}')
print(f'Email matches yours  : {"✅" if email_ok else "❌  MISMATCH — fix YOUR_EMAIL in Cell 1"}')
print(f'CORS header (*)      : {"✅" if cors_ok else "❌"}')

if version_ok and email_ok and cors_ok:
    print(f'\n🎉 ALL CHECKS PASSED! Submit this URL to the exam (Q18):')
    print(f'\n   👉  {public_url}')
    print('\n   ⚠️  Submit the BASE ngrok URL only — do NOT add /api/version')
else:
    print('\n⚠️  Fix the issues above and re-run from Cell 7.')

print('\n--- Stop commands ---')
print('import os')
print(f'os.killpg(os.getpgid(ngrok_proc.pid), 9)   # stop ngrok')
print(f'os.killpg(os.getpgid(proxy_proc.pid), 9)   # stop proxy')
print(f'os.killpg(os.getpgid(ollama_proc.pid), 9)  # stop ollama')

---
## 📋 Quick Reference

### Q10 — what to submit
```
https://xyz.trycloudflare.com/api
```
The checker calls `/api?class=1A&class=2B...` — returns students from your CSV in original row order.

### Q18 — what to submit
```
https://abcd1234.ngrok-free.app
```
Submit the **base** ngrok URL only (no `/api/version`).

### Why each student's solution is different
| | Q10 | Q18 |
|--|-----|-----|
| **Unique part** | Your `q-fastapi.csv` | `X-Email` header value |
| **Same for everyone** | Server code, `/api` logic | Ollama install, proxy structure |
| **What to change** | Upload your own CSV | Set `YOUR_EMAIL` in Cell 1 |

# Q25 Guide — Deploy a POST Analytics Endpoint to Vercel



## ✅ Dynamic Solution Guide — Works for Every Student

> **What makes Q25 unique per student:**
> The exam generates your personal telemetry JSON (36 records) using your email as a seed.
> The `threshold_ms` and which 2 regions are tested also differ per student.
> You must embed **your own JSON** in the deployed endpoint — the shared repo is just a template.

---

## Step 1 — Download Your Telemetry JSON

Go to the exam portal → Q25 → click **"Download telemetry bundle"**.

Save it as `telemetry.json`. Open it and note:
- It has 36 records across 3 regions: `apac`, `emea`, `amer`
- Each record has: `region`, `service`, `latency_ms`, `uptime_pct`, `timestamp`

---

## Step 2 — Clone the Shared Repo

```bash
git clone https://github.com/GyaanFlow/t22026-tds-ga0-q25.git
cd t22026-tds-ga0-q25
```

---

## Step 3 — Update `api/index.py` with Your JSON

Open `api/index.py`. Find the `TELEMETRY_DATA` list and **replace it entirely** with your downloaded JSON.

Your file should look like this (paste your own data in the middle):

```python
from fastapi import FastAPI, Request
from fastapi.middleware.cors import CORSMiddleware
from fastapi.responses import Response
import numpy as np
import json

app = FastAPI()

app.add_middleware(
    CORSMiddleware,
    allow_origins=["*"],
    allow_credentials=False,
    allow_methods=["*"],
    allow_headers=["*"],
    expose_headers=["*"],
)

@app.get("/")
def root():
    return {"status": "ok"}

@app.options("/api/latency")
async def options_handler():
    return Response(status_code=200)

# ✏️  PASTE YOUR OWN TELEMETRY JSON HERE (downloaded from exam portal)
TELEMETRY_DATA = json.loads("""
[
  {"region": "apac", "service": "support", "latency_ms": 147.13, ...},
  ...  ← your 36 rows go here
]
""")

@app.post("/api/latency")
async def latency_analytics(request: Request):
    body = await request.json()
    regions = body.get("regions", [])
    threshold_ms = body.get("threshold_ms", 180)

    results = []
    for region in regions:
        records   = [r for r in TELEMETRY_DATA if r["region"] == region]
        latencies = [r["latency_ms"] for r in records]
        uptimes   = [r["uptime_pct"]  for r in records]
        results.append({
            "region":      region,
            "avg_latency": round(float(np.mean(latencies)), 2),
            "p95_latency": round(float(np.percentile(latencies, 95)), 2),
            "avg_uptime":  round(float(np.mean(uptimes)), 3),
            "breaches":    int(sum(1 for l in latencies if l > threshold_ms))
        })

    return {"regions": results}
```

> ⚠️ **Do not change the calculation logic** — only replace the `TELEMETRY_DATA` block.
> Everyone's logic is the same; only the data rows differ.

---

## Step 4 — Verify the Repo Has These Files

```
t22026-tds-ga0-q25/
├── api/
│   └── index.py        ← updated with YOUR telemetry data
├── vercel.json
└── requirements.txt
```

`vercel.json` should contain:
```json
{
  "builds": [{ "src": "api/index.py", "use": "@vercel/python" }],
  "routes": [{ "src": "/(.*)", "dest": "api/index.py" }]
}
```

`requirements.txt` should contain:
```
fastapi
numpy
```

---

## Step 5 — Push Your Changes to GitHub

```bash
git add api/index.py
git commit -m "Update telemetry data for my exam submission"
git push
```

> ⚠️ Push to **your own fork** of the repo, not the shared one.
> If you cloned directly (not forked), create your own GitHub repo and push there:
> ```bash
> git remote set-url origin https://github.com/YOUR_USERNAME/YOUR_REPO.git
> git push -u origin main
> ```

---

## Step 6 — Deploy to Vercel

### Option A — Vercel Website (easiest)

1. Go to [vercel.com](https://vercel.com) → **New Project**
2. Import your GitHub repo
3. Leave all settings default → click **Deploy**
4. Wait ~1 min → copy the deployment URL (e.g. `https://your-app.vercel.app`)

### Option B — Vercel CLI (from Colab or terminal)

```bash
# Install
npm install -g vercel

# Deploy (follow prompts — link to your GitHub repo)
vercel --prod
```

---

## Step 7 — Test Before Submitting

Run this in Colab to verify your endpoint returns correct values:

```python
import requests

VERCEL_URL = "https://your-app.vercel.app/api/latency"  # ← paste your URL

# Use the regions and threshold from YOUR exam question
payload = {
    "regions": ["apac", "emea"],   # ← change to your 2 regions
    "threshold_ms": 175            # ← change to your threshold
}

r = requests.post(VERCEL_URL, json=payload)
print("Status :", r.status_code)
print("CORS   :", r.headers.get("access-control-allow-origin"))
print("Body   :", r.json())
```

Expected response shape:
```json
{
  "regions": [
    {
      "region": "apac",
      "avg_latency": 168.45,
      "p95_latency": 219.12,
      "avg_uptime": 98.519,
      "breaches": 4
    },
    ...
  ]
}
```

The exam validator checks (tolerances):
| Field | Tolerance |
|-------|-----------|
| `avg_latency` | ±0.5 ms |
| `p95_latency` | ±0.5 ms |
| `avg_uptime` | ±0.2 % |
| `breaches` | exact match |

---

## Step 8 — Submit to Exam

Paste the **full `/api/latency` endpoint URL** — not the base domain:

| ❌ Wrong | ✅ Correct |
|---------|-----------|
| `https://your-app.vercel.app` | `https://your-app.vercel.app/api/latency` |
| `https://your-app.vercel.app/` | `https://your-app.vercel.app/api/latency` |

> ✅ The hostname **must contain** `vercel.app`
> ✅ Must end with `/api/latency`
> ✅ CORS header must be `Access-Control-Allow-Origin: *`
> ✅ Must accept `POST` with JSON body

---

## Summary — What Changes Per Student

| Part | Same for everyone | What YOU change |
|------|------------------|-----------------|
| `api/index.py` logic | ✅ Yes | Replace `TELEMETRY_DATA` with your JSON |
| `vercel.json` | ✅ Yes | Nothing |
| `requirements.txt` | ✅ Yes | Nothing |
| Deployed URL | ❌ No | Your Vercel URL |
| Telemetry data | ❌ No | Downloaded from exam portal |
| Test regions & threshold | ❌ No | Shown in your exam Q25 |